# 04.2.4 - Supervised Learning: Klasifikasi Akurasi Tertinggi dengan ANN (K-Means QLDE)

## 1. Tujuan Analisis
Sebagai penutup eksperimen pada dataset hasil **K-Means QLDE**, kita akan mengerahkan algoritma yang paling kompleks dan canggih: **Artificial Neural Network (ANN)** atau Jaringan Saraf Tiruan.

Seperti yang sudah kita bahas sebelumnya, ANN bekerja layaknya "otak tiruan" atau sebuah pabrik dengan banyak lapisan pegawai spesialis (*hidden layers*). Tujuan kita di sini adalah melihat batas maksimal akurasi yang bisa dicapai oleh mesin untuk mengenali pola kelompok pelanggan hasil QLDE. Apakah "otak tiruan" ini mampu memecahkan tantangan di *Class 5* yang sebelumnya menyulitkan model SVM dan AdaBoost?

In [1]:
import pandas as pd
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

# 1. BACA DATA (Gunakan Pemenang Clustering: QLDE)
df = pd.read_csv('../data/Labeled/hasildata_kmeans-qlde.csv')

# Pisahkan Fitur (Var1 - Var11) dan Target (Cluster)
fitur = [f'Var{i}' for i in range(1, 12)]
X = df[fitur]
y = df['Cluster']

# Splitting Data: 80% Train, 20% Test (Menggunakan seluruh baris data)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 2. Proses Scaling (Wajib untuk ANN)
Sama seperti algoritma SVM, algoritma ANN sangat sensitif terhadap perbedaan besaran angka. Agar "pegawai" di dalam ANN tidak kebingungan dan mengira angka yang besar (seperti total transaksi) jauh lebih penting daripada angka yang kecil (seperti jumlah hari), kita wajib menyetarakan angkanya menggunakan `StandardScaler`.

In [2]:
# 2. SCALING DATA
scaler_ann = StandardScaler()

# Model scaler belajar dari data latih (Train) sekaligus menyetarakan angkanya
X_train_scaled = scaler_ann.fit_transform(X_train)

# Data ujian (Test) juga disetarakan agar formatnya seragam
X_test_scaled = scaler_ann.transform(X_test)

## 3. Training Model ANN
Kita mulai melatih modelnya dengan menyewa 100 "pegawai spesialis" di ruang tengah (`hidden_layer_sizes=(100,)`) dan memberikan waktu belajar maksimal sebanyak 500 putaran (`max_iter=500`).

*Catatan:* Jika saat dijalankan muncul peringatan kotak berwarna merah/pink (`ConvergenceWarning`), itu sama sekali bukan *error*. Itu hanya pesan dari mesin yang merasa dirinya masih bisa belajar lebih lama lagi, namun terhenti karena batas 500 putaran kita. Hasil tebakannya pun sudah sangat maksimal.

In [3]:
# 3. TRAINING MODEL ANN
# Menggunakan max_iter=500 agar model punya cukup waktu belajar dan konvergen
model_ann = MLPClassifier(hidden_layer_sizes=(100,), max_iter=500, random_state=42)
model_ann.fit(X_train_scaled, y_train)

# Evaluasi model menggunakan data ujian
prediksi_ann = model_ann.predict(X_test_scaled)
print("=== CLASSIFICATION REPORT: ANN (QLDE) ===\n")
print(classification_report(y_test, prediksi_ann))

=== CLASSIFICATION REPORT: ANN (QLDE) ===

              precision    recall  f1-score   support

           0       0.98      0.98      0.98       150
           1       0.96      0.99      0.97       214
           2       0.98      0.98      0.98       125
           3       0.98      0.96      0.97        85
           4       0.98      0.96      0.97       199
           5       0.91      0.91      0.91        94

    accuracy                           0.97       867
   macro avg       0.97      0.96      0.96       867
weighted avg       0.97      0.97      0.97       867



c:\Users\USER\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


### Analisis Performa (QLDE vs Standard)
Sangat memuaskan! ANN berhasil meraih akurasi sebesar **97%** (atau `0.97`) pada data QLDE. Ia bahkan berhasil mengenali *Class 5* dengan akurasi tebakan benar (*precision* & *recall*) mencapai 91%, jauh lebih baik dibanding model-model sebelumnya.

Jika kita bandingkan dengan performa ANN pada data K-Means Standard (yang mencapai 98%), hasilnya beda tipis. Ini mengonfirmasi bahwa sebaran data hasil K-Means QLDE memang sedikit lebih rumit, namun otak tiruan ANN terbukti tetap tangguh dan berhasil memetakannya dengan nyaris sempurna.

## 4. Ekspor Model dan Scaler
Kita wajib menyimpan alat penyetara angka dan mesin otak tiruannya secara berpasangan agar bisa dipanggil kembali kapan saja tanpa harus *training* ulang.

In [4]:
# 4. EXPORT SCALER & MODEL KE FOLDER 'models'
os.makedirs('../models', exist_ok=True)

# Simpan Scaler
joblib.dump(scaler_ann, '../models/scaler_ann_qlde.pkl')

# Simpan Model ANN
joblib.dump(model_ann, '../models/model_ann_classification_qlde.pkl')

print("\n[SUCCESS] Scaler & Model ANN berhasil diekspor ke folder '../models/'!")


[SUCCESS] Scaler & Model ANN berhasil diekspor ke folder '../models/'!
